In [ ]:
#setup
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
pd.options.display.precision = 4
rng = np.random.default_rng(42)
print(sys.executable)

In [ ]:
#a p-value by brute force
#A coin lands heads 60 times in 100 flips. Is it unfair? H₀: p = 0.5.

observed = 60
sim = rng.binomial(100, 0.5, size=1_000_000)              # a million experiments with a FAIR coin
p_two = np.mean(np.abs(sim - 50) >= abs(observed - 50))   # as far from 50 as 60 is, in either direction
p_one = np.mean(sim >= observed)

print(f"simulated: two-sided p = {p_two:.4f}, one-sided p = {p_one:.4f}")
print(f"exact    : two-sided p = {stats.binomtest(observed, 100, 0.5).pvalue:.4f}, "
      f"one-sided p = {stats.binomtest(observed, 100, 0.5, alternative='greater').pvalue:.4f}")

In [ ]:
#a z-test, with its picture
def z_test(xbar, mu0, sigma, n, alternative="two-sided"):
    z = (xbar - mu0) / (sigma / math.sqrt(n))
    if alternative == "two-sided":
        p = 2 * stats.norm.sf(abs(z))
    elif alternative == "greater":
        p = stats.norm.sf(z)
    elif alternative == "less":
        p = stats.norm.cdf(z)
    else:
        raise ValueError("alternative must be 'two-sided', 'greater' or 'less'")
    return z, p

def plot_z(z_obs, alpha=0.05, alternative="two-sided"):
    x = np.linspace(-4, 4, 800)
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.plot(x, stats.norm.pdf(x), color="black")
    if alternative == "two-sided":
        c = stats.norm.ppf(1 - alpha / 2)
        reject, tail = [x <= -c, x >= c], [x <= -abs(z_obs), x >= abs(z_obs)]
    elif alternative == "greater":
        c = stats.norm.ppf(1 - alpha)
        reject, tail = [x >= c], [x >= z_obs]
    else:
        c = stats.norm.ppf(alpha)
        reject, tail = [x <= c], [x <= z_obs]
    for r in reject:
        ax.fill_between(x[r], stats.norm.pdf(x[r]), color="tomato", alpha=0.35)
    for t in tail:
        ax.fill_between(x[t], stats.norm.pdf(x[t]), color="steelblue", alpha=0.45)
    ax.axvline(z_obs, color="black", ls="--")
    ax.set_title(f"{alternative}: red = rejection region (alpha = {alpha}), blue = the p-value, dashed = observed z")
    plt.show()

# A machine should fill bottles to 500 ml (sigma = 4 ml is known). 36 bottles average 498.8 ml.
for alt in ("two-sided", "less"):
    z, p = z_test(498.8, 500, 4, 36, alt)
    print(f"{alt:<10}: z = {z:.3f}, p = {p:.4f} -> {'reject H0' if p <= 0.05 else 'fail to reject H0'}")
plot_z(-1.8, alternative="two-sided")
plot_z(-1.8, alternative="less")

In [ ]:
#tests for proportions
# One proportion: 540 of 900 people support a measure. H0: p = 0.5
n, x, p0 = 900, 540, 0.5
p_hat = x / n
z = (p_hat - p0) / math.sqrt(p0 * (1 - p0) / n)           # the SE uses the HYPOTHESISED p0
print(f"one proportion: p-hat = {p_hat:.2f}, z = {z:.2f}, two-sided p = {2 * stats.norm.sf(abs(z)):.2e}")

# Two proportions (an A/B test): 120 of 1000 convert on design A, 150 of 1000 on design B
def two_prop_z(x1, n1, x2, n2):
    p1, p2 = x1 / n1, x2 / n2
    pooled = (x1 + x2) / (n1 + n2)                         # under H0 both groups share one rate
    se = math.sqrt(pooled * (1 - pooled) * (1 / n1 + 1 / n2))
    z = (p2 - p1) / se
    return z, 2 * stats.norm.sf(abs(z))

z, p = two_prop_z(120, 1000, 150, 1000)
print(f"two proportions: 12.0% vs 15.0%, z = {z:.3f}, p = {p:.4f}")

In [ ]:
#Type I error and power by simulation
def p_values(true_mean, n=36, sigma=4, reps=50_000, mu0=500):
    x = rng.normal(true_mean, sigma, size=(reps, n))
    z = (x.mean(axis=1) - mu0) / (sigma / math.sqrt(n))
    return 2 * stats.norm.sf(np.abs(z))

def power_z(delta, sigma, n, alpha=0.05):
    shift = delta / (sigma / math.sqrt(n))
    c = stats.norm.ppf(1 - alpha / 2)
    return stats.norm.sf(c - shift) + stats.norm.cdf(-c - shift)

print(f"H0 true (mean 500): rejected {(p_values(500) < 0.05).mean():.3f} of the time   <- the Type I error rate, close to alpha = 0.05")
for true_mean in (499, 498, 497):
    power = (p_values(true_mean) < 0.05).mean()
    print(f"true mean {true_mean}: power {power:.3f} (formula {power_z(500 - true_mean, 4, 36):.3f}), Type II error {1 - power:.3f}")

In [ ]:
#what controls power, and how much data you need
fig, ax = plt.subplots(figsize=(7, 3.6))
ns = np.arange(5, 400)
for delta in (0.5, 1, 2):
    ax.plot(ns, [power_z(delta, 4, n) for n in ns], label=f"true difference = {delta}")
ax.axhline(0.8, color="gray", ls="--")
ax.set_xlabel("sample size n")
ax.set_ylabel("power")
ax.legend()
plt.show()

def n_for_power(delta, sigma, power=0.8, alpha=0.05):
    z_a, z_b = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    return math.ceil(((z_a + z_b) * sigma / delta) ** 2)

print("n for 80% power to detect a difference of 1 (sigma 4):", n_for_power(1, 4))
print("n for 90% power:", n_for_power(1, 4, power=0.9), "| a difference of 0.5:", n_for_power(0.5, 4))

In [ ]:
#what a p-value looks like when H₀ is true, and when it isn’t
reps, n = 20_000, 30
p_null = stats.ttest_1samp(rng.normal(0, 1, size=(reps, n)), 0, axis=1).pvalue
p_alt = stats.ttest_1samp(rng.normal(0.5, 1, size=(reps, n)), 0, axis=1).pvalue

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
axes[0].hist(p_null, bins=20, range=(0, 1), color="lightsteelblue", edgecolor="white")
axes[0].set_title("H0 true: p-values are UNIFORM")
axes[1].hist(p_alt, bins=20, range=(0, 1), color="lightgreen", edgecolor="white")
axes[1].set_title("true effect of half an SD: p-values pile up near 0")
for ax in axes:
    ax.axvline(0.05, color="tomato")
plt.show()
print(f"H0 true : P(p < 0.05) = {(p_null < 0.05).mean():.3f}")
print(f"H1 true : P(p < 0.05) = {(p_alt < 0.05).mean():.3f}   <- the power (about 0.75)")



#When there is no effect, every p-value is equally likely, so 5% of the p-values fall below 0.05 by pure chance. That one fact explains the next three traps.

In [ ]:
#trap 1, multiple testing
reps, tests, n = 10_000, 20, 30
p = stats.ttest_1samp(rng.normal(0, 1, size=(reps, tests, n)), 0, axis=2).pvalue    # 20 tests of pure noise

print(f"P(at least one 'significant' result in 20 tests) = {(p < 0.05).any(axis=1).mean():.3f}   (formula {1 - 0.95 ** 20:.3f})")
print(f"with Bonferroni (alpha / 20 = 0.0025)           = {(p < 0.05 / tests).any(axis=1).mean():.3f}")

In [ ]:
#trap 2, peeking and stopping early
x = rng.normal(0, 1, size=(5000, 200))                      # 5,000 experiments where H0 is true
ever_significant = np.zeros(5000, dtype=bool)
for k in range(10, 201, 10):                                 # peek after every 10 observations
    ever_significant |= stats.ttest_1samp(x[:, :k], 0, axis=1).pvalue < 0.05
print(f"stopping the moment p < 0.05 gives a 'discovery' in {ever_significant.mean():.3f} of the experiments (should be 0.05)")

In [ ]:
#trap 3, statistical vs practical significance
huge = rng.normal(100.1, 15, size=1_000_000)                # a true difference of 0.1 on a scale with SD 15
res = stats.ttest_1samp(huge, 100)
d = (huge.mean() - 100) / huge.std(ddof=1)
print(f"p = {res.pvalue:.1e}, mean difference = {huge.mean() - 100:.3f}, Cohen's d = {d:.4f}")

In [ ]:
#a battery’s claimed life is 10 hours
battery = np.array([9.2, 9.8, 10.1, 8.7, 9.5, 9.9, 10.3, 9.0, 9.4, 9.6, 8.9, 9.7])
mu0 = 10
n, xbar, s = len(battery), battery.mean(), battery.std(ddof=1)
se = s / math.sqrt(n)
t_stat, df = (xbar - mu0) / se, n - 1
p_two = 2 * stats.t.sf(abs(t_stat), df)
print(f"by hand: mean {xbar:.4f}, s {s:.4f}, SE {se:.4f}, t = {t_stat:.3f}, df = {df}, two-sided p = {p_two:.4f}")

res = stats.ttest_1samp(battery, mu0)
print(f"scipy  : t = {res.statistic:.3f}, p = {res.pvalue:.4f}")
print("left-tailed p:", round(stats.ttest_1samp(battery, mu0, alternative="less").pvalue, 4))

low, high = stats.t.interval(0.95, df, loc=xbar, scale=se)
print(f"95% CI for the mean: ({low:.3f}, {high:.3f})  -> does it contain {mu0}? {low <= mu0 <= high}")
print(f"effect size d = {(xbar - mu0) / s:.2f}")
print(f"if we wrongly used z: p = {2 * stats.norm.sf(abs(t_stat)):.4f}  (the t-test's p is ten times larger: small samples need the fatter tails)")

In [ ]:
#the t distribution
x = np.linspace(-5, 5, 600)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(x, stats.norm.pdf(x), color="black", label="normal")
for df_ in (2, 5, 30):
    ax.plot(x, stats.t.pdf(x, df_), label=f"t, df = {df_}")
ax.legend()
plt.show()
print("P(|T| > 3) for df = 5 :", round(2 * stats.t.sf(3, 5), 4), "| for the normal:", round(2 * stats.norm.sf(3), 4))

In [ ]:
#a non-parametric alternative
print(stats.wilcoxon(battery - mu0))                        # Wilcoxon signed-rank test: no normality needed

In [ ]:
#two teaching methods
A = np.array([68, 72, 75, 70, 66, 74, 71, 69, 73, 70])
B = np.array([74, 78, 80, 72, 77, 75, 79, 73, 76, 81])

va, vb = A.var(ddof=1) / len(A), B.var(ddof=1) / len(B)
se = math.sqrt(va + vb)
t_stat = (A.mean() - B.mean()) / se
df = (va + vb) ** 2 / (va ** 2 / (len(A) - 1) + vb ** 2 / (len(B) - 1))       # Welch-Satterthwaite
print(f"by hand: means {A.mean():.1f} and {B.mean():.1f}, SE {se:.3f}, t = {t_stat:.3f}, df = {df:.2f}, p = {2 * stats.t.sf(abs(t_stat), df):.5f}")

print("Welch :", stats.ttest_ind(A, B, equal_var=False))
print("pooled:", stats.ttest_ind(A, B))                                          # the DEFAULT is the pooled version

def cohens_d(a, b):
    a, b = np.asarray(a), np.asarray(b)
    pooled_sd = math.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / pooled_sd

def welch_ci(a, b, level=0.95):
    a, b = np.asarray(a), np.asarray(b)
    va, vb = a.var(ddof=1) / len(a), b.var(ddof=1) / len(b)
    df = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
    diff, t_star = a.mean() - b.mean(), stats.t.ppf((1 + level) / 2, df)
    half = t_star * math.sqrt(va + vb)
    return diff, (diff - half, diff + half)

print(f"Cohen's d = {cohens_d(A, B):.2f}")
diff, (lo, hi) = welch_ci(A, B)
print(f"difference A - B = {diff:.1f}, 95% CI ({lo:.2f}, {hi:.2f})")

fig, ax = plt.subplots(figsize=(5, 3.6))
ax.boxplot([A, B])
ax.set_xticks([1, 2])
ax.set_xticklabels(["method A", "method B"])
plt.show()
print("non-parametric check:", stats.mannwhitneyu(A, B, alternative="two-sided"))

In [ ]:
#why Welch is the safer default
def type1_rate(n1, sd1, n2, sd2, equal_var, reps=20_000):
    a = rng.normal(0, sd1, size=(reps, n1))                 # both groups have the SAME mean, so H0 is true
    b = rng.normal(0, sd2, size=(reps, n2))
    return (stats.ttest_ind(a, b, axis=1, equal_var=equal_var).pvalue < 0.05).mean()

print("false-positive rate (should be 0.05)")
for label, n1, sd1, n2, sd2 in (("equal sizes, equal SD", 30, 5, 30, 5),
                                ("equal sizes, unequal SD", 30, 5, 30, 15),
                                ("small group has the BIG variance", 10, 10, 100, 1)):
    print(f"  {label:<34} pooled {type1_rate(n1, sd1, n2, sd2, True):.3f}   Welch {type1_rate(n1, sd1, n2, sd2, False):.3f}")

In [ ]:
#from summary statistics only
res = stats.ttest_ind_from_stats(mean1=82, std1=10, nobs1=40, mean2=78, std2=12, nobs2=50, equal_var=False)
print(res)

In [ ]:
#blood pressure before and after a treatment
before = np.array([150, 142, 160, 155, 148, 165, 152, 158])
after = np.array([144, 140, 152, 150, 147, 158, 150, 151])
diff = after - before

n, dbar, sd = len(diff), diff.mean(), diff.std(ddof=1)
se = sd / math.sqrt(n)
t_stat = dbar / se
print(f"by hand: mean change {dbar:.2f}, SD of the changes {sd:.3f}, SE {se:.3f}, t = {t_stat:.3f}, df = {n - 1}, "
      f"p = {2 * stats.t.sf(abs(t_stat), n - 1):.4f}")

print("paired      :", stats.ttest_rel(after, before))
print("same thing  :", stats.ttest_1samp(diff, 0))
print("WRONG test  :", stats.ttest_ind(after, before, equal_var=False))
print("correlation between before and after:", round(np.corrcoef(before, after)[0, 1], 3))
print("95% CI for the mean change:", np.round(stats.t.interval(0.95, n - 1, loc=dbar, scale=se), 2))
print("Wilcoxon signed-rank:", stats.wilcoxon(after, before))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for b, a in zip(before, after):
    axes[0].plot([0, 1], [b, a], "o-", color="gray")
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["before", "after"])
axes[0].set_title("each line is one person")
axes[1].hist(diff, bins=6, color="lightsteelblue", edgecolor="white")
axes[1].axvline(0, color="tomato")
axes[1].set_title("the differences")
plt.tight_layout()
plt.show()

In [ ]:
#Mini project, an A/B test

#2A website tested two page designs. Analysis of both outcomes, as a report.

#the data

n = 300
time_a = rng.lognormal(mean=3.4, sigma=0.5, size=n)         # seconds on the page, design A
time_b = rng.lognormal(mean=3.5, sigma=0.5, size=n)         # design B
converted = {"A": (36, 300), "B": (51, 300)}                 # signed up: (successes, visitors)

In [ ]:
#the analysis
print(pd.DataFrame({"A": pd.Series(time_a).describe(), "B": pd.Series(time_b).describe()}).round(1))
print("skew:", round(stats.skew(time_a), 2), round(stats.skew(time_b), 2))

# Time on page: skewed data, but n = 300 per group, so the CLT makes Welch fine. Check on the log scale too
for label, a, b in (("raw seconds", time_a, time_b), ("log seconds", np.log(time_a), np.log(time_b))):
    res = stats.ttest_ind(a, b, equal_var=False)
    diff, (lo, hi) = welch_ci(a, b)
    print(f"{label:<12}: difference A - B = {diff:.3f}, 95% CI ({lo:.3f}, {hi:.3f}), p = {res.pvalue:.4f}, d = {cohens_d(a, b):.2f}")
print("Mann-Whitney:", round(stats.mannwhitneyu(time_a, time_b).pvalue, 4))

# Sign-up rate
(x1, n1), (x2, n2) = converted["A"], converted["B"]
z, p = two_prop_z(x1, n1, x2, n2)
print(f"\nsign-ups: A {x1 / n1:.1%}, B {x2 / n2:.1%}, z = {z:.3f}, p = {p:.4f}")

# How much data would a reliable test have needed?
print("n per group for 80% power, to detect a real 10-point improvement of 12% to 22%:",
      math.ceil(((1.96 + 0.8416) ** 2 * (0.12 * 0.88 + 0.22 * 0.78) / 0.10 ** 2)))


#the sign-up test gives z ≈ −1.74 and p ≈ 0.082. B looks better but the evidence isn’t conclusive,
# which is typical of a test that was underpowered (300 per group).
# That is not proof that the designs are equal.

## A/B test report

**Question.** Does design B make visitors stay longer, or sign up more often, than design A?

**Methods.** (Which tests, which assumptions you checked, alpha = 0.05, chosen in advance.)

**Findings.**
- Time on page: A averaged __ s, B averaged __ s. The difference is __ s (95% CI __ to __), p = __.
- Sign-ups: A __%, B __%. The difference is __ points, p = __.

**Meaning.** (Plain words. Distinguish "no evidence of a difference" from "evidence of no difference".)

**Limits.** (Sample size, how visitors were assigned, how long the test ran, the number of metrics tested.)

**Recommendation.** (Ship B / keep A / run a bigger test, and how big.)